# Deposit Attrition — v8b · corrections to the v8 run

Three corrections, one of them a defect in v8's own code.

### 1 · The institution field is rail-determined, and the rail is ACH

§4c printed "spread across rails" because the verdict tested a 60% **wire** threshold. The data says
something sharper. `has_fin` is **1.0000 or 0.0000 for every outbound rail** — perfectly
deterministic — and the 37% decomposes exactly:

`ACH 0.3584 × 1.000` + `RTP_PRT 0.0102 × 0.996` + `WIRE 0.0012 × 1.000` = **0.3698** against a
reported overall of 0.3700. Sixty-three points are uncovered because **59 of them are debit card**,
a rail that structurally cannot carry a receiving institution.

`fin_out_n` is *the number of distinct institutions a client sends ACH to*. Not a bank count, and
not — as v8 hypothesised — a wire signal: §4f shows it performs **better** among non-wire clients
(AUC 0.743) than among wire users (0.676), where balance beats it outright.

### 2 · The coverage ceiling is the dd construction, not the field

§4d: **74.3%** of customer-months carry a `fin_out_n` value. Only 31–42% carry a `dd`. The dd
construction costs ~43 points of coverage — far more than any field choice. That is where the
coverage work is, and §1 below is the first instalment.

### 3 · `MIN_REF = 1.0` silently deleted five feature families — v8's own defect

`chg = value / mean(t−6…t−4)` is gated on `ref > 1.0`, which is right for counts and amounts and
**impossible for a bounded share**. Every ratio feature came back `dd` null on every row:
`conc_top*`, `railmix_*`, every `*_retention`, every `*_rec_broken_share`, `acc_closed_share`.

That is why `+ concentration` and `+ railmix` reported `n_feat = 36`, identical to the baseline.
**Those blocks added zero columns.** Their 0.0000 deltas are not evidence; they were never tested.

## What this notebook does

| § | |
|---|---|
| 1 | Additive dd for bounded ratios — difference from own baseline, then difference from peer. Sanity check moves from `median_dd = 1.000` to `median_dd = 0.000` |
| 2 | Re-ablate the five nulled families, now that they carry data |
| 3 | **Combination specs.** `+ fin_in` won on precision (0.6514 → 0.7143). Test it against small combinations rather than against "everything" |
| 4 | **Regularisation sweep.** "+ everything" had the best AUC and the worst precision — 206 features on 313k rows at `L2 = 2.0`. Test whether that is overfitting or a real ceiling |
| 5 | The shipping recommendation |

Reuses `features_v8`, the v7 risk set and the v6 peer anchor. Nothing is rebuilt from the payment
table.

In [ ]:
# =====================================================================
# 0 · SETUP — reuses the v8 kernel's config if present
# =====================================================================
import warnings, time, math, numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
from pathlib import Path
warnings.filterwarnings("ignore")

if "spark" not in dir():
    spark = (SparkSession.builder.appName("pkg_attrition_eda_v8b")
             .config("spark.sql.shuffle.partitions", "800")
             .config("spark.sql.execution.arrow.pyspark.enabled", "false")
             .enableHiveSupport().getOrCreate())
if "HDFS_DIR" not in dir():
    HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
    HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
    HDFS_V7  = "hdfs://nameservice1/user/pk36814/attrition_v7"
    HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v8"
    OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v8")
    def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
    def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
    def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
    def v7(n): return f"{HDFS_V7.rstrip('/')}/{n}"
    CHG_LAG_FAR, CHG_LAG_NEAR, CHG_MIN_OBS, MIN_REF = -6, -4, 2, 1.0
    PEER_MIN_N, DD_CLIP, SEED = 50, (0.01, 100.0), 20260907
    ORIGIN_START_OFF, MAX_ORIGINS, PRIMARY_H = 18, 24, 6
    HORIZONS, NEG_SAMPLE, MIN_TRAIN_POS = [1, 3, 6], 0.15, 300
    CAPACITY, QUEUE_K, L2 = [50, 100, 250, 500, 1000, 2500], 250, 2.0
    IRLS_MAX_IT, IRLS_TOL, MAX_COLLECT_ROWS = 60, 1e-9, 3_000_000
    PRIMARY_DEF = "A_full_exit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── the correction: which features are BOUNDED RATIOS ─────────────────
# A multiplicative dd needs a reference mean above MIN_REF=1.0. A share
# lives in [0,1] and can never clear it, so every one of these came back
# null on every row in v8. They get an ADDITIVE dd instead:
#   chg = value - mean(t-6..t-4)        (difference from own baseline)
#   dd  = chg  - median(chg over peers) (difference from peer drift)
# Centred at 0, not 1. The sanity check flips accordingly.
def is_ratio(f):
    return (f.endswith(("_share", "_retention", "_ratio")) or
            f in ("conc_top1", "conc_top3", "railmix_shift",
                  "railmix_rails_collapsed", "tim_day_shift", "tim_day_sd"))

L2_GRID = [2.0, 20.0, 100.0, 400.0]
print("v8b ready")


In [ ]:
# =====================================================================
# 1 · ADDITIVE dd FOR BOUNDED RATIOS                     [OUTPUT BLOCK 1]
# =====================================================================
def disp(o, title=None, n=60, save=None):
    out = o.limit(n).toPandas() if hasattr(o, "toPandas") else pd.DataFrame(o)
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title: display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                           f"margin:10px 0 2px'>{title}</div>"))
    display(out); return out
def kv(pairs, title=None, save=None):
    items = list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)

t0 = time.time()
V8F = spark.read.parquet(hp("features_v8")).persist(StorageLevel.DISK_ONLY)
RS7 = spark.read.parquet(v7("risk_set"))
anchor = spark.read.parquet(v6("peer_anchor")).select("cust_pwr_id", "peer_key")
cust_month = spark.read.parquet(v2("panel_customer_month"))
YMMAP = cust_month.select("ym", "m_idx").distinct().persist(StorageLevel.DISK_ONLY)
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]

NEWCOLS = [c for c in V8F.columns if c not in ("cust_pwr_id", "m_idx")]
RATIO = [c for c in NEWCOLS if is_ratio(c)]
COUNTY = [c for c in NEWCOLS if not is_ratio(c)]
print(f"  {len(RATIO)} bounded-ratio features rebuilt additively, {len(COUNTY)} left as ratios")

_stack = ", ".join([f"'{f}', CAST(`{f}` AS DOUBLE)" for f in RATIO])
lg = (V8F.select("cust_pwr_id", "m_idx", *RATIO)
      .join(anchor, "cust_pwr_id", "left").join(YMMAP, "m_idx", "left")
      .select("cust_pwr_id", "m_idx", "ym", "peer_key",
              F.expr(f"stack({len(RATIO)}, {_stack}) as (feature, value)")))
wr = (Window.partitionBy("cust_pwr_id", "feature").orderBy("m_idx")
      .rangeBetween(CHG_LAG_FAR, CHG_LAG_NEAR))     # rangeBetween, never rowsBetween
lg = (lg.withColumn("ref", F.avg("value").over(wr))
        .withColumn("nref", F.count("value").over(wr))
        # ADDITIVE. No MIN_REF gate — that gate is what nulled these in v8.
        .withColumn("chg", F.when((F.col("nref") >= CHG_MIN_OBS) & F.col("value").isNotNull(),
                                  F.col("value") - F.col("ref"))))
pc = (lg.groupBy("ym", "peer_key", "feature")
      .agg(F.count("chg").alias("pn"), F.expr("percentile_approx(chg, 0.5)").alias("pchg"))
      .filter(F.col("pn") >= PEER_MIN_N))
lg = (lg.join(pc, ["ym", "peer_key", "feature"], "left")
        .withColumn("dd", F.when(F.col("chg").isNotNull() & F.col("pchg").isNotNull(),
                                 F.col("chg") - F.col("pchg"))))

CHK = disp(lg.groupBy("feature").agg(
        F.avg(F.col("dd").isNotNull().cast("double")).alias("share_with_dd"),
        F.expr("percentile_approx(dd, 0.5)").alias("median_dd")).orderBy("feature"),
     title="1a &middot; Additive dd on the bounded ratios. <b>The sanity check is now "
           "median_dd = 0.000, not 1.000</b> — and share_with_dd was 0.0000 for every one of "
           "these in v8", n=40, save="v8b_ratio_dd")

# NAMING TRAP: pivot output is {pivotValue}_{aggAlias} only with TWO OR MORE
# aggregations. v8 had two (dd and v) so {f}_dd was right there. With a single
# agg the column comes back as the bare pivot value — {f}, not {f}_dd.
wide = (lg.groupBy("cust_pwr_id", "m_idx").pivot("feature", RATIO)
        .agg(F.first("dd", True)))
assert all(f in wide.columns for f in RATIO), (
    f"pivot naming: expected bare feature names, got {wide.columns[:6]}")
sel = [F.col("cust_pwr_id"), F.col("m_idx")]
for f in RATIO:
    dd = F.col(f)
    # additive dd is already centred and unbounded-ish; clip symmetrically
    # rather than log — log of a signed difference is undefined
    sel.append(F.when(dd.isNotNull(), F.greatest(F.least(dd, F.lit(5.0)), F.lit(-5.0)))
                .alias(f"ld_{f}"))
    sel.append(F.when(dd.isNull(), F.lit(1.0)).otherwise(F.lit(0.0)).alias(f"md_{f}"))
RFIX = wide.select(*sel)

RISK8 = spark.read.parquet(hp("risk_set_v8"))
DROP = [c for f in RATIO for c in (f"ld_{f}", f"md_{f}", f"tr_{f}") if c in RISK8.columns]
RISK8B = RISK8.drop(*DROP).join(RFIX, ["cust_pwr_id", "m_idx"], "left")
RISK8B.write.mode("overwrite").partitionBy("m_idx").parquet(hp("risk_set_v8b"))
RISK8B = spark.read.parquet(hp("risk_set_v8b")).persist(StorageLevel.DISK_ONLY)
kv([("bounded-ratio features repaired", len(RATIO)),
    ("mean dd coverage, was", 0.0),
    ("mean dd coverage, now", round(float(CHK.share_with_dd.mean()), 4)),
    ("risk-set columns", len(RISK8B.columns)),
    ("wall (s)", round(time.time()-t0))],
   title="1b &middot; The repair", save="v8b_repair")


In [ ]:
# =====================================================================
# 2 · RE-ABLATION + COMBINATIONS + L2 SWEEP              [OUTPUT BLOCK 2]
# =====================================================================
def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y)-n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))
def logit_irls(X, y, l2, mi=IRLS_MAX_IT, tol=IRLS_TOL):
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    for _ in range(mi):
        eta = np.clip(X @ b, -30, 30); mu = 1/(1+np.exp(-eta))
        w = np.maximum(mu*(1-mu), 1e-6); z = eta + (y-mu)/w; XtW = X.T*w
        try: bn = np.linalg.solve(XtW @ X + R, XtW @ z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW @ X + R, XtW @ z, rcond=None)[0]
        if np.max(np.abs(bn-b)) < tol: b = bn; break
        b = bn
    return b
def fit_spec(tr, cols, l2=L2, s=NEG_SAMPLE):
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck: return None
    Xk = X[:, keep]; mu = Xk.mean(0); sd = Xk.std(0)
    b = logit_irls(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, l2)
    return dict(cols=ck, beta=b[1:]/sd,
                b0=float(b[0]-float(np.sum(b[1:]*mu/sd))) + math.log(s))
def apply_spec(sp, df):
    return np.full(len(df), np.nan) if sp is None else \
           sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
def topk(score, y, ks, pb=None):
    s = np.asarray(score, float); y = np.asarray(y, float); n = len(s)
    fin = np.isfinite(s); s = np.where(fin, s, -np.inf)
    ys = y[np.argsort(-s, kind="stable")]; tot = y.sum()
    pb = (tot/n) if pb is None else pb
    return pd.DataFrame([dict(k=K, tp=int(ys[:min(K, n)].sum()),
                              precision=ys[:min(K, n)].sum()/min(K, n),
                              recall=ys[:min(K, n)].sum()/tot if tot else np.nan)
                         for K in ks])

ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - PRIMARY_H + 1))
assert 1 <= len(ORIGINS) <= MAX_ORIGINS, f"{len(ORIGINS)} origins — m_idx is ABSOLUTE"
ALLC = RISK8B.columns
NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
        "selfpay", "acc_")
V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                and not any(c.startswith("ld_"+p) for p in NEWP)])
V7_MD = [f"md_{c[3:]}" for c in V7_LD if f"md_{c[3:]}" in ALLC]
def blk(*pfx):
    ld = sorted([c for c in ALLC if any(c.startswith("ld_"+p) for p in pfx)])
    return ld + [f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]
B = {"railmix": blk("railmix"), "concentration": blk("conc_"), "accounts": blk("acc_"),
     "cpty_name_out": blk("cptyn_out"), "cpty_acct_out": blk("cptya_out"),
     "cpty_in": blk("cptyn_in"), "fin_in": blk("fin_in"), "fin_out2": blk("fin_out2"),
     "timing": blk("tim_"), "selfpay": blk("selfpay"),
     "recurring": sorted([c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))])}
B = {k: v for k, v in B.items() if v}
disp(pd.DataFrame([dict(block=k, n=len(v)) for k, v in B.items()]),
     title="2a &middot; Blocks after the repair. <b>railmix, concentration and accounts carried "
           "zero usable columns in v8</b>", save="v8b_blocks")

NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B"] + V7_LD + V7_MD +
                  [c for v in B.values() for c in v]) & set(ALLC))
ip = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)) |
      F.col("event_B").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)))
def cpd(sdf, lab):
    n = sdf.count(); assert n <= MAX_COLLECT_ROWS, f"{lab}: {n:,}"
    t = time.time(); o = sdf.toPandas(); print(f"  {lab}: {n:,} in {time.time()-t:,.0f}s"); return o
TR = cpd(RISK8B.filter(F.col("m_idx") <= max(ORIGINS) - min(HORIZONS))
         .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                     F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
         .filter(ip | (F.col("_u") < NEG_SAMPLE)).select(*NEED), "TRAIN")
TE = {t: cpd(RISK8B.filter(F.col("m_idx") == t).select(*NEED), f"TEST {t}") for t in ORIGINS}

def prep(d):
    d = d.copy()
    for c in d.columns:
        if c.startswith("ld_"):   d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
        elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    return d
def label(d, H=PRIMARY_H):
    ev = pd.to_numeric(d["event_A"], errors="coerce"); t = pd.to_numeric(d["m_idx"], errors="coerce")
    y = ((ev > t) & (ev <= t+H)).astype(float); obs = (t+H <= M_MAX) | (y == 1)
    o = d.loc[obs].copy(); o["y"] = y.loc[obs].values; return o
TRp = prep(TR); TEp = {k: prep(v) for k, v in TE.items()}

BASE = V7_LD + V7_MD
SPECS = {"v7_baseline": (BASE, L2)}
for k, v in B.items(): SPECS[f"+ {k}"] = (BASE + v, L2)
# combinations, built up from the v8 winner rather than from "everything"
SPECS["+ fin_in + recurring"]                 = (BASE + B["fin_in"] + B["recurring"], L2)
SPECS["+ fin_in + cpty_acct_out"]             = (BASE + B["fin_in"] + B["cpty_acct_out"], L2)
SPECS["+ fin_in + recurring + cpty_acct_out"] = (BASE + B["fin_in"] + B["recurring"] +
                                                 B["cpty_acct_out"], L2)
ALLB = sorted({c for v in B.values() for c in v})
for l2 in L2_GRID:
    SPECS[f"+ everything (L2={l2:g})"] = (BASE + ALLB, l2)

rows, tk = [], []
for Tm in ORIGINS:
    tr = label(TRp[TRp.m_idx <= Tm - PRIMARY_H]); te = label(TEp[Tm])
    if tr.y.sum() < MIN_TRAIN_POS or te.y.sum() < 1: continue
    pb = float(te.y.mean())
    for nm, (cols, l2) in SPECS.items():
        cols = [c for c in cols if c in tr.columns]
        sp = fit_spec(tr, cols, l2); s = apply_spec(sp, te)
        rows.append(dict(spec=nm, origin=Tm, auc=auc(te.y.values, s),
                         n_feat=len(sp["cols"]) if sp else 0))
        m = topk(s, te.y.values, CAPACITY, pb); m.insert(0, "spec", nm); tk.append(m)
AB = pd.DataFrame(rows); TK = pd.concat(tk, ignore_index=True)
P = TK.groupby(["spec", "k"], as_index=False).agg(tp=("tp", "sum"), alerts=("k", "sum"),
                                                  recall=("recall", "mean"))
P["precision"] = P.tp/P.alerts
SUM = (AB.groupby("spec", as_index=False).agg(auc=("auc", "mean"), sd=("auc", "std"),
                                              n_feat=("n_feat", "max"))
       .merge(P[P.k == QUEUE_K][["spec", "precision", "recall"]], on="spec"))
b_a = float(SUM.loc[SUM.spec == "v7_baseline", "auc"].iloc[0])
b_p = float(SUM.loc[SUM.spec == "v7_baseline", "precision"].iloc[0])
SUM["d_auc"] = (SUM.auc - b_a).round(4); SUM["d_prec"] = (SUM.precision - b_p).round(4)
# PRECISION at the operating capacity is the ship test. AUC is a global
# measure and the queue lives in the extreme tail — v8 showed the two
# disagreeing hard on "+ everything".
SUM["ships"] = np.where(SUM.d_prec > 0.010, "yes",
                np.where(SUM.spec == "v7_baseline", "&mdash;", "no"))
disp(SUM.sort_values("d_prec", ascending=False).round(4),
     title=f"2b &middot; <b>Ranked on precision at K={QUEUE_K}</b>, not AUC. Baseline "
           f"{b_a:.4f} / {b_p:.4f}", n=40, save="v8b_ablation")


In [ ]:
# =====================================================================
# 3 · THE SHIPPING DECISION                              [OUTPUT BLOCK 3]
# =====================================================================
E = SUM[SUM.spec.str.startswith("+ everything")].copy()
E["L2"] = E.spec.str.extract(r"L2=([0-9.]+)").astype(float)
disp(E[["L2", "n_feat", "auc", "precision", "d_auc", "d_prec"]].sort_values("L2").round(4),
     title="3a &middot; <b>Was &lsquo;everything&rsquo; overfitting?</b> Same 206 features, "
           "rising ridge. If precision climbs with L2 while AUC holds, v8's precision collapse "
           "was capacity, not signal", save="v8b_l2_sweep")

best = SUM.sort_values("d_prec", ascending=False).iloc[0]
_fin = SUM[SUM.spec == "+ fin_in"]
_ev2 = E.sort_values("precision", ascending=False).iloc[0] if len(E) else None
kv([("baseline precision at K", round(b_p, 4)),
    ("best spec on precision", str(best.spec)),
    ("  ...its precision", round(float(best.precision), 4)),
    ("  ...its AUC", round(float(best.auc), 4)),
    ("  ...features", int(best.n_feat)),
    ("fin_in alone, precision", round(float(_fin.precision.iloc[0]), 4) if len(_fin) else None),
    ("best 'everything', L2", float(_ev2.L2) if _ev2 is not None else None),
    ("best 'everything', precision", round(float(_ev2.precision), 4) if _ev2 is not None else None),
    ("blocks that ship", int((SUM.ships == "yes").sum())),
    ("repaired blocks that now ship",
     int(SUM[SUM.spec.isin([f"+ {k}" for k in ("railmix", "concentration", "accounts")])]
         .ships.eq("yes").sum()))],
   title="3b &middot; Verdict", save="v8b_verdict")

SHIP = SUM[(SUM.ships == "yes") & (~SUM.spec.str.startswith("+ everything"))]
disp(SHIP.sort_values("d_prec", ascending=False)[["spec", "n_feat", "auc", "precision",
                                                  "recall", "d_auc", "d_prec"]].round(4),
     title="3c &middot; Everything that earns its place, ranked. Anything absent gets "
           "<b>deleted, not parked</b>", save="v8b_ship_list")


---

## Read this before adding customer attributes

1. **Ship on precision, not AUC.** v8 made the case by itself: `+ everything` had the best AUC
   (0.7973) and the worst precision of any shipping block (0.5977). A queue lives in the extreme
   tail of the ranking and AUC is a global average. If §3a shows precision recovering as `L2`
   rises, the fix is regularisation; if it does not, the large spec is genuinely worse and the
   small one ships.

2. **Restate `fin_out_n` everywhere.** It is *the number of distinct institutions a client sends
   ACH to*. `has_fin` is 1.0000 on ACH, RTP_PRT and WIRE and 0.0000 on card, cheque, PCARD and
   RTP_P2P — deterministic, not sparse. Its 37% ceiling is 59 points of debit card plus rounding,
   and no field choice moves it. The v7/v8 briefs both need this line.

3. **The dd construction is the coverage problem now.** 74.3% of customer-months carry a
   `fin_out_n` value; 31% carry a `dd`. Before adding attributes it is worth asking whether the
   `t−6…t−4` reference window and the `MIN_REF` gate are the right defaults for every feature
   family, or whether a shorter reference and a level-plus-change pair would keep more of the book
   scoreable. That is a bigger prize than any single new field.

4. **Drop the name-key recommendation.** Account key +0.0052, name key +0.0031, and name-keyed
   coverage is *lower* (0.4639 vs 0.5036) because the generic registry strips 40.7% of links. If
   the name key is revisited, rebuild the registry **on the study population** — the current one
   was fitted on the whole bank and its top entries (`MOM`, `DAD`, `JOSE RODRIGUEZ`, `CHECKING`)
   are retail P2P strings that a Treasury client would never pay.

5. **Then bring segment, NAICS and size.** Measure them against whatever §3c settles on, not
   against v7 — otherwise their lift and the payment rebuild's are inseparable.
